# Module 3 Exercise: Warehouse Restore Points

Fabric Warehouse does not expose `BACKUP DATABASE` / `RESTORE DATABASE`. Instead,
every commit is a Delta commit, and you can pin a named **restore point** on top
of the transaction log, then roll back to it with one call.

SQL Server analogues:

| Warehouse operation                | SQL Server equivalent                         |
|------------------------------------|-----------------------------------------------|
| `create_restore_point`             | `BACKUP DATABASE ... WITH COPY_ONLY`          |
| Restore to a named point           | `RESTORE DATABASE ... FROM DISK ... WITH ...` |
| Underlying mechanism               | Delta log versioning (no physical copy)       |

The call below uses [`semantic-link-labs`](https://github.com/microsoft/semantic-link-labs),
the supported Python library for Fabric control-plane operations. It runs from
any Fabric notebook and talks to the workspace REST APIs under the hood.


## 1. Install `semantic-link-labs`

One-off install per notebook session.


In [ ]:
%pip install semantic-link-labs


## 2. Create a named restore point on the `gold` warehouse

Think of this as a *logical checkpoint*, not a backup file. Nothing is copied,
Delta just tags the current log version with your name and description so you
can find it again later.

Leave `workspace=None` (the default) to use the current workspace. Override only
if you are targeting a different workspace.


In [ ]:
from sempy_labs.warehouse import create_restore_point

create_restore_point(
    warehouse="gold",
    name="Before deployment",
    description="Restore point before release",
)


## 3. What to check next

- In the Fabric portal, open the `gold` warehouse and look under
  **Settings → Restore points**. The named point appears immediately.
- Run a destructive statement (e.g. `DELETE FROM dbo.customers`), then use the
  portal to restore to the point you just created. The table returns.

## Key takeaway

Restore points are the Warehouse's answer to backups: instantaneous, free, and
driven by the same Delta log you learned about in Module 1. The mental model is
point-in-time, not file-based. One call, no storage cost, no maintenance plan.
